# Bài tập 3 — Luật kết hợp (Association Rules)

## Bộ dữ liệu D3: US Accidents

### Mục tiêu

Khai phá các luật kết hợp trong dữ liệu tai nạn giao thông Hoa Kỳ nhằm tìm ra:

- Các điều kiện (thời tiết, khung giờ, đặc điểm hạ tầng) thường xuất hiện đồng thời trong một vụ tai nạn.
- Mối liên hệ giữa điều kiện xảy ra tai nạn và mức độ nghiêm trọng (`Severity`).
- Các luật có mức độ liên kết đáng chú ý dựa trên Support, Confidence và Lift.

Quy trình thực hiện:

1. Chuẩn bị dữ liệu giao dịch.
2. Rời rạc hóa và giảm không gian item.
3. Nhị phân hóa dữ liệu giao dịch.
4. Khai phá frequent itemsets bằng Apriori và FP-Growth.
5. Sinh luật kết hợp.
6. Đánh giá bằng Support, Confidence và Lift.
7. Lọc các luật tầm thường, không phù hợp hoặc dư thừa.
8. Phân tích một số luật nổi bật từ kết quả thực tế.

> **Notebook này sử dụng dữ liệu đầu vào là `accidents_clean.csv` — kết quả đã được khảo sát, kiểm tra thiếu/ngoại lệ và làm sạch ở Bài 1 (`khao-satD3.ipynb`). Bài 3 không đọc lại dữ liệu thô (`US_Accidents_sampled.csv`) và không lặp lại các bước kiểm tra đã thực hiện ở Bài 1; toàn bộ phần bên dưới chỉ tập trung vào bước biến đổi đặc thù của luật kết hợp (gộp `Weather_Condition`, rời rạc hóa thời gian, định nghĩa giao dịch, nhị phân hóa) và khai phá luật — theo đúng định dạng đã dùng ở bộ D2 (`luat-ket-hop-D2.ipynb`).**

> **`accidents_clean.csv` có 195,845 dòng (sau khi Bài 1 loại các dòng thiếu `Severity`/`Weather_Condition`/`Sunrise_Sunset`), từ mẫu 200,000 dòng đầu của `US_Accidents_sampled.csv`. Vì vậy, các kết quả frequent itemsets và association rules được hiểu là kết quả trên mẫu dữ liệu này.**


# 1. Chuẩn bị dữ liệu giao dịch

Ở Bài 1 (`khao-satD3.ipynb`), dữ liệu thô `US_Accidents_sampled.csv` đã được khảo sát, kiểm tra thiếu/ngoại lệ, tách đặc trưng thời gian (`start_hour`, `start_month`, `start_weekday`, `duration_minutes`) và làm sạch thành bảng chi tiết **`accidents_clean.csv`** — chỉ loại các dòng thiếu `Severity`, `Weather_Condition` hoặc `Sunrise_Sunset` (ba cột dùng làm "neo" để tạo item). Bài 3 đọc trực tiếp file này làm điểm xuất phát.

Khác với bộ D2 (một đơn hàng gồm nhiều sản phẩm nên cần `groupby` để dựng basket), ở D3 **mỗi dòng của `accidents_clean.csv` đã tương ứng với một vụ tai nạn**. Trong bài toán luật kết hợp, cần chuyển mỗi dòng thành:

`transaction → {item1, item2, item3, ...}`

Trong đó mỗi transaction tương ứng với một vụ tai nạn, và mỗi item là một điều kiện có mặt tại vụ tai nạn đó (mức độ nghiêm trọng, nhóm thời tiết, sáng/tối, khung giờ, loại ngày, đặc điểm hạ tầng).


In [1]:
# ĐỌC DỮ LIỆU ĐÃ LÀM SẠCH

import pandas as pd
import numpy as np
import time

from pathlib import Path
from IPython.display import display

# Tìm thư mục gốc của project (đồng bộ cách tìm ROOT với khao-satD3.ipynb)
current = Path.cwd()

while current != current.parent:
    if (current / "data" / "processed" / "D3_us_accidents").exists():
        break
    current = current.parent

ROOT = current
PROCESSED = ROOT / "data" / "processed" / "D3_us_accidents"

# accidents_clean.csv: bảng chi tiết đã làm sạch ở Bài 1
df = pd.read_csv(PROCESSED / "accidents_clean.csv")

print(f"df: {df.shape}")
display(df.head())


df: (195845, 18)


,ID,Severity,Weather_Condition,Sunrise_Sunset,start_hour,start_month,start_weekday,duration_minutes,Distance(mi),Temperature(F),Visibility(mi),Amenity,Bump,Crossing,Junction,Railway,Stop,Traffic_Signal
0,A-75728,2,Clear,Day,16,11,0,44.583333,0.01,62.1,10.0,0,0,0,0,0,0,0
1,A-80191,3,Clear,Day,12,9,2,29.700000,0.00,91.9,10.0,0,0,0,0,0,0,0
2,A-19865,3,Clear,Night,20,9,1,30.000000,0.00,66.2,10.0,0,0,0,0,0,0,0
3,A-76706,3,Overcast,Day,10,9,1,30.000000,0.00,64.9,10.0,0,0,0,0,0,0,0
4,A-92998,3,Clear,Day,17,8,5,30.000000,0.00,72.0,10.0,0,0,0,0,0,0,0


## 1.1. Định nghĩa transaction / basket

Trong bộ dữ liệu US Accidents, `ID` (mã vụ tai nạn) được chọn làm định danh transaction.

Mỗi vụ tai nạn được xem như một **giao dịch**, và các điều kiện đồng thời xuất hiện tại vụ tai nạn đó (mức độ nghiêm trọng, thời tiết, sáng/tối, khung giờ, loại ngày, đặc điểm hạ tầng xung quanh) tạo thành một **basket**.

Biểu diễn:

$$ Transaction = \{condition_1, condition_2, ..., condition_n\} $$

Ví dụ một vụ tai nạn xảy ra vào buổi tối, trời nhiều mây, gần đèn tín hiệu giao thông:

$$ Accident_1 = \{Severity\_2,\ Weather\_Cloudy,\ Time\_Evening,\ Traffic\_Signal\} $$

thì đây được xem là một transaction có bốn item.

Việc dùng mỗi vụ tai nạn làm một transaction phù hợp với mục tiêu bài toán: tìm ra các **tổ hợp điều kiện** (thời tiết, thời điểm, hạ tầng) thường xuất hiện cùng nhau và cùng một mức độ nghiêm trọng, tương tự cách Market Basket Analysis tìm các sản phẩm thường được mua cùng nhau.


In [2]:
print("=" * 60)
print("TRANSACTION / BASKET")
print("=" * 60)

n_transactions = df["ID"].nunique()

print(f"Số transaction (vụ tai nạn): {n_transactions:,}")

print("\nPhân bố Severity:")
display(
    df["Severity"]
    .value_counts()
    .sort_index()
    .rename_axis("Severity")
    .reset_index(name="accident_count")
)


TRANSACTION / BASKET
Số transaction (vụ tai nạn): 195,845

Phân bố Severity:


,Severity,accident_count
0,1,1897
1,2,133985
2,3,57440
3,4,2523


## 1.2. Gộp `Weather_Condition` thành nhóm

Cột `Weather_Condition` có tới hàng chục giá trị khác nhau (nhiều giá trị chỉ xuất hiện vài lần, ví dụ "Heavy T-Storm / Windy", "Squalls"...). Nếu giữ nguyên toàn bộ danh mục, không gian tìm kiếm itemset của Apriori/FP-Growth sẽ tăng không cần thiết vì phần lớn các nhóm hiếm gần như không bao giờ đạt ngưỡng support tối thiểu.

Do đó, `Weather_Condition` được gộp thành:

- 10 nhóm thời tiết phổ biến nhất (giữ nguyên tên).
- Toàn bộ các nhóm còn lại được gộp vào **`Khac`**.

Đây là bước **gộp nhóm/aggregation theo danh mục** (không phải chia một biến số thành các khoảng), tương tự cách bộ D2 gộp `product_id → aisle`.

#### **Việc gộp này giúp:**
- Giảm số lượng item liên quan đến thời tiết.
- Giảm độ thưa (nhiều nhóm hiếm chỉ chiếm vài chục giao dịch).
- Tăng khả năng xuất hiện đồng thời của các item thời tiết trong frequent itemsets.
- Làm luật kết hợp dễ diễn giải hơn.


In [3]:
# GỘP WEATHER_CONDITION THÀNH NHÓM

TOP_K_WEATHER = 10

top_weather = (
    df["Weather_Condition"]
    .value_counts()
    .head(TOP_K_WEATHER)
    .index
)

df["weather_group"] = np.where(
    df["Weather_Condition"].isin(top_weather),
    df["Weather_Condition"],
    "Khac"
)

coverage = (
    df["Weather_Condition"].isin(top_weather).mean()
)

print("=" * 60)
print("GỘP WEATHER_CONDITION THÀNH NHÓM")
print("=" * 60)

print(f"Số giá trị Weather_Condition gốc : {df['Weather_Condition'].nunique():,}")
print(f"Số nhóm sau khi gộp (kể cả Khac) : {df['weather_group'].nunique():,}")
print(f"Tỷ lệ giao dịch thuộc {TOP_K_WEATHER} nhóm phổ biến nhất: {coverage:.2%}")

print("\nPhân bố nhóm thời tiết:")
display(
    df["weather_group"]
    .value_counts()
    .rename_axis("weather_group")
    .reset_index(name="accident_count")
)


GỘP WEATHER_CONDITION THÀNH NHÓM
Số giá trị Weather_Condition gốc : 94
Số nhóm sau khi gộp (kể cả Khac) : 11
Tỷ lệ giao dịch thuộc 10 nhóm phổ biến nhất: 94.27%

Phân bố nhóm thời tiết:


,weather_group,accident_count
0,Fair,46139
1,Clear,35733
2,Mostly Cloudy,27270
3,Partly Cloudy,18884
4,Overcast,16713
5,Cloudy,16022
6,Khac,11226
7,Light Rain,9726
8,Scattered Clouds,9077
9,Light Snow,2830


## 1.3. Rời rạc hóa thuộc tính số

Bộ dữ liệu có các thuộc tính số liên quan đến thời gian:

- `start_hour`: giờ xảy ra tai nạn, từ 0 đến 23.
- `start_weekday`: thứ trong tuần (0 = Thứ Hai ... 6 = Chủ Nhật).

Các giá trị số này không thể trực tiếp biểu diễn như một item trong basket. Vì vậy cần rời rạc hóa thành các nhóm.

### Rời rạc hóa `start_hour`

Chia thành 4 khoảng, sử dụng cùng quy tắc đã dùng ở bộ D2 để dễ so sánh giữa hai bộ dữ liệu:

| Khoảng giờ | Nhãn |
|---|---|
| 00–05 | `Time_Night` |
| 06–11 | `Time_Morning` |
| 12–17 | `Time_Afternoon` |
| 18–23 | `Time_Evening` |

### Rời rạc hóa `start_weekday`

`start_weekday` được chuyển thành:

- `Weekday_Accident`: xảy ra vào thứ Hai – thứ Sáu.
- `Weekend_Accident`: xảy ra vào thứ Bảy, Chủ Nhật.

Sau rời rạc hóa, các giá trị này có thể được xem như item ngữ cảnh trong basket, bên cạnh `Sunrise_Sunset` (`Day`/`Night`) vốn đã là thuộc tính hạng mục nhị phân, không cần rời rạc hóa thêm.

> **Basket cuối cùng gồm ba nhóm item: (1) mức độ nghiêm trọng và nhóm thời tiết, (2) các item ngữ cảnh được rời rạc hóa từ thời gian (`time_period`, `day_type`) cùng `Sunrise_Sunset` có sẵn, và (3) các đặc điểm hạ tầng xung quanh (`Amenity`, `Crossing`, `Traffic_Signal`...). Nhờ đó, thuật toán có thể phát hiện quan hệ giữa điều kiện thời tiết/thời gian, đặc điểm hạ tầng và mức độ nghiêm trọng của tai nạn.**


In [4]:
# ------------------------------------------------------------
# Rời rạc hóa giờ xảy ra tai nạn
# ------------------------------------------------------------

def classify_time(hour):
    if 0 <= hour < 6:
        return "Time_Night"
    elif 6 <= hour < 12:
        return "Time_Morning"
    elif 12 <= hour < 18:
        return "Time_Afternoon"
    else:
        return "Time_Evening"


df["time_period"] = df["start_hour"].apply(classify_time)


# ------------------------------------------------------------
# Rời rạc hóa thứ trong tuần
# ------------------------------------------------------------

df["day_type"] = np.where(
    df["start_weekday"].isin([5, 6]),
    "Weekend_Accident",
    "Weekday_Accident"
)


print("=" * 60)
print("RỜI RẠC HÓA THUỘC TÍNH SỐ")
print("=" * 60)

print("\nPhân bố khung giờ:")
display(
    df["time_period"]
    .value_counts()
    .rename_axis("time_period")
    .reset_index(name="accident_count")
)

print("\nPhân bố loại ngày:")
display(
    df["day_type"]
    .value_counts()
    .rename_axis("day_type")
    .reset_index(name="accident_count")
)

print("\nPhân bố sáng/tối (Sunrise_Sunset, không cần rời rạc hóa thêm):")
display(
    df["Sunrise_Sunset"]
    .value_counts()
    .rename_axis("Sunrise_Sunset")
    .reset_index(name="accident_count")
)


RỜI RẠC HÓA THUỘC TÍNH SỐ

Phân bố khung giờ:


,time_period,accident_count
0,Time_Morning,83082
1,Time_Afternoon,66066
2,Time_Evening,30980
3,Time_Night,15717



Phân bố loại ngày:


,day_type,accident_count
0,Weekday_Accident,174176
1,Weekend_Accident,21669



Phân bố sáng/tối (Sunrise_Sunset, không cần rời rạc hóa thêm):


,Sunrise_Sunset,accident_count
0,Day,144065
1,Night,51780


## 1.4. Xây dựng basket

Mỗi transaction (vụ tai nạn) được biểu diễn bằng danh sách item gồm:

- `Severity_<1..4>` — mức độ nghiêm trọng (nhị phân hóa hạng mục từ `Severity`).
- `Weather_<nhóm>` — nhóm thời tiết sau khi gộp ở mục 1.2.
- `Day` / `Night` — lấy trực tiếp từ `Sunrise_Sunset`.
- `time_period` — khung giờ sau khi rời rạc hóa `start_hour`.
- `day_type` — loại ngày sau khi rời rạc hóa `start_weekday`.
- Các cờ hạ tầng (`Amenity`, `Bump`, `Crossing`, `Junction`, `Railway`, `Stop`, `Traffic_Signal`) — chỉ thêm vào basket nếu giá trị bằng 1 tại vụ tai nạn đó.

Cách làm này tương tự bộ D2 (aisle + time_period + day_type), chỉ khác là D3 có thêm các item nhị phân về hạ tầng, và số item mỗi transaction có thể thay đổi tùy vào số cờ hạ tầng đang bật (0–7 cờ), cộng với 5 item cố định (severity, weather, sáng/tối, khung giờ, loại ngày).


In [5]:
FLAG_COLS = [
    "Amenity", "Bump", "Crossing", "Junction",
    "Railway", "Stop", "Traffic_Signal"
]


def build_basket(row):
    items = [
        f"Severity_{int(row['Severity'])}",
        f"Weather_{row['weather_group']}",
        row["Sunrise_Sunset"],
        row["time_period"],
        row["day_type"],
    ]

    for col in FLAG_COLS:
        if row[col] == 1:
            items.append(col)

    return items


df["basket"] = df.apply(build_basket, axis=1)

df["basket_size"] = df["basket"].apply(len)

print("=" * 60)
print("BASKET SAU KHI RỜI RẠC HÓA VÀ GỘP NHÓM")
print("=" * 60)

display(df[["ID", "Severity", "weather_group", "Sunrise_Sunset",
            "time_period", "day_type", "basket"]].head(10))

print("\nKích thước basket:")
display(df["basket_size"].describe())


BASKET SAU KHI RỜI RẠC HÓA VÀ GỘP NHÓM


,ID,Severity,weather_group,Sunrise_Sunset,time_period,day_type,basket
0,A-75728,2,Clear,Day,Time_Afternoon,Weekday_Accident,"[Severity_2, Weather_Clear, Day, Time_Afternoo..."
1,A-80191,3,Clear,Day,Time_Afternoon,Weekday_Accident,"[Severity_3, Weather_Clear, Day, Time_Afternoo..."
2,A-19865,3,Clear,Night,Time_Evening,Weekday_Accident,"[Severity_3, Weather_Clear, Night, Time_Evenin..."
3,A-76706,3,Overcast,Day,Time_Morning,Weekday_Accident,"[Severity_3, Weather_Overcast, Day, Time_Morni..."
4,A-92998,3,Clear,Day,Time_Afternoon,Weekend_Accident,"[Severity_3, Weather_Clear, Day, Time_Afternoo..."
5,A-76441,2,Mostly Cloudy,Day,Time_Morning,Weekday_Accident,"[Severity_2, Weather_Mostly Cloudy, Day, Time_..."
6,A-84011,3,Clear,Day,Time_Afternoon,Weekday_Accident,"[Severity_3, Weather_Clear, Day, Time_Afternoo..."
7,A-80924,3,Clear,Day,Time_Afternoon,Weekend_Accident,"[Severity_3, Weather_Clear, Day, Time_Afternoo..."
8,A-60768,2,Overcast,Night,Time_Evening,Weekday_Accident,"[Severity_2, Weather_Overcast, Night, Time_Eve..."
9,A-50075,3,Partly Cloudy,Day,Time_Afternoon,Weekday_Accident,"[Severity_3, Weather_Partly Cloudy, Day, Time_..."



Kích thước basket:


count    195845.000000
mean          5.451204
std           0.726570
min           5.000000
25%           5.000000
50%           5.000000
75%           6.000000
max          10.000000
Name: basket_size, dtype: float64

## 1.5. Nhị phân hóa

Sau khi chuẩn bị basket, dữ liệu có dạng:

$$ Transaction \rightarrow \{item_1,item_2,...,item_n\} $$

Để đưa dữ liệu vào Apriori và FP-Growth, cần chuyển sang ma trận:

$$ Transaction \times Item $$

Sử dụng `TransactionEncoder` của thư viện `mlxtend`.

Quy ước:

- `True`: item xuất hiện trong transaction (điều kiện có mặt tại vụ tai nạn).
- `False`: item không xuất hiện.

Ví dụ:

| Transaction | Severity_2 | Crossing | Time_Evening |
|---|---:|---:|---:|
| A-75728 | True | False | False |
| A-80191 | True | False | False |

Ma trận này là đầu vào cho bước khai phá frequent itemsets.


In [6]:
# NHỊ PHÂN HÓA BẰNG TRANSACTIONENCODER

from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()

encoded_array = te.fit(
    df["basket"]
).transform(
    df["basket"]
)

basket_df = pd.DataFrame(
    encoded_array,
    columns=te.columns_,
    index=df["ID"]
)

print("=" * 60)
print("MA TRẬN GIAO DỊCH NHỊ PHÂN")
print("=" * 60)

print(f"Số transaction: {basket_df.shape[0]:,}")
print(f"Số item: {basket_df.shape[1]:,}")
print(f"Kích thước: {basket_df.shape}")

display(basket_df.head())


MA TRẬN GIAO DỊCH NHỊ PHÂN
Số transaction: 195,845
Số item: 30
Kích thước: (195845, 30)


,Amenity,Bump,Crossing,Day,Junction,Night,Railway,Severity_1,Severity_2,Severity_3,...,Weather_Khac,Weather_Light Rain,Weather_Light Snow,Weather_Mostly Cloudy,Weather_Overcast,Weather_Partly Cloudy,Weather_Rain,Weather_Scattered Clouds,Weekday_Accident,Weekend_Accident
ID,,,,,,,,,,,,,,,,,,,,,
A-75728,False,False,False,True,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,True,False
A-80191,False,False,False,True,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,True,False
A-19865,False,False,False,False,False,True,False,False,False,True,...,False,False,False,False,False,False,False,False,True,False
A-76706,False,False,False,True,False,False,False,False,False,True,...,False,False,False,False,True,False,False,False,True,False
A-92998,False,False,False,True,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,True


## 1.6. Output kiểm tra

Kiểm tra dữ liệu sau khi hoàn thành toàn bộ quá trình chuẩn bị:

- Số transaction.
- Số item.
- Kích thước basket.
- Giá trị thiếu.
- Item trùng trong basket.
- Kích thước basket nhỏ nhất/lớn nhất/trung bình.
- Các giá trị xuất hiện trong ma trận.
- Tổng số giá trị `True`.

Dữ liệu chỉ được đưa sang bước khai phá khi ma trận có dạng nhị phân hợp lệ và không còn giá trị thiếu.


In [7]:
# ============================================================
# 1.6. OUTPUT KIỂM TRA DỮ LIỆU
# ============================================================

print("=" * 70)
print("KIỂM TRA DỮ LIỆU SAU KHI CHUẨN BỊ")
print("=" * 70)


# ------------------------------------------------------------
# 1. Transaction
# ------------------------------------------------------------

n_transactions = df["ID"].nunique()


# ------------------------------------------------------------
# 2. Item
# ------------------------------------------------------------

n_items = len(te.columns_)


# ------------------------------------------------------------
# 3. Kích thước matrix
# ------------------------------------------------------------

matrix_rows, matrix_cols = basket_df.shape


# ------------------------------------------------------------
# 4. Missing values
# ------------------------------------------------------------

missing_values = basket_df.isna().sum().sum()


# ------------------------------------------------------------
# 5. Item trùng trong basket
# ------------------------------------------------------------

duplicate_items = (
    df["basket"]
    .apply(lambda x: len(x) != len(set(x)))
    .sum()
)


# ------------------------------------------------------------
# 6. Basket size
# ------------------------------------------------------------

basket_sizes = basket_df.sum(axis=1)


# ------------------------------------------------------------
# 7. Kiểm tra giá trị matrix
# ------------------------------------------------------------

unique_values = pd.unique(basket_df.values.ravel())
is_binary = set(unique_values).issubset({True, False})


# ------------------------------------------------------------
# 8. Tổng số item xuất hiện
# ------------------------------------------------------------

total_true = int(basket_df.sum().sum())


# ============================================================
# HIỂN THỊ KẾT QUẢ
# ============================================================

print("\n--- TỔNG QUAN ---")
print(f"Số transaction           : {n_transactions:,}")
print(f"Số item                  : {n_items:,}")
print(f"Kích thước basket matrix : ({matrix_rows:,}, {matrix_cols:,})")
print(f"Số giá trị thiếu         : {missing_values:,}")
print(f"Basket có item trùng     : {duplicate_items:,}")

print("\n--- KÍCH THƯỚC BASKET ---")
print(f"Nhỏ nhất                 : {basket_sizes.min():.0f}")
print(f"Lớn nhất                 : {basket_sizes.max():.0f}")
print(f"Trung bình               : {basket_sizes.mean():.2f}")
print(f"Trung vị                 : {basket_sizes.median():.2f}")

print("\n--- KIỂM TRA MA TRẬN NHỊ PHÂN ---")
print(f"Các giá trị xuất hiện   : {unique_values}")
print(f"Tổng số giá trị True    : {total_true:,}")

print("\n" + "=" * 70)
print("KẾT LUẬN")
print("=" * 70)

if (
    n_transactions > 0
    and n_items > 0
    and missing_values == 0
    and duplicate_items == 0
    and is_binary
):
    print("✓ Dữ liệu hợp lệ.")
    print("✓ Basket đã được nhị phân hóa.")
    print("✓ Sẵn sàng cho Apriori và FP-Growth.")
else:
    print("⚠ Dữ liệu chưa hợp lệ.")
    print("⚠ Cần kiểm tra lại trước khi khai phá.")


KIỂM TRA DỮ LIỆU SAU KHI CHUẨN BỊ

--- TỔNG QUAN ---
Số transaction           : 195,845
Số item                  : 30
Kích thước basket matrix : (195,845, 30)
Số giá trị thiếu         : 0
Basket có item trùng     : 0

--- KÍCH THƯỚC BASKET ---
Nhỏ nhất                 : 5
Lớn nhất                 : 10
Trung bình               : 5.45
Trung vị                 : 5.00

--- KIỂM TRA MA TRẬN NHỊ PHÂN ---
Các giá trị xuất hiện   : [False  True]
Tổng số giá trị True    : 1,067,591

KẾT LUẬN
✓ Dữ liệu hợp lệ.
✓ Basket đã được nhị phân hóa.
✓ Sẵn sàng cho Apriori và FP-Growth.


# 2. Thuật toán Apriori và FP-Growth

Sau khi xây dựng ma trận basket dạng nhị phân, tiến hành khai thác các tập mục phổ biến (frequent itemsets) bằng hai thuật toán:

- **Apriori**
- **FP-Growth**

Hai mức `min_support` được sử dụng, giữ cùng giá trị với bộ D2 để tiện so sánh giữa hai bộ dữ liệu:

- `0.01`: ngưỡng cao, giúp giảm số lượng itemset và luật sinh ra.
- `0.005`: ngưỡng thấp hơn, cho phép phát hiện thêm các itemset ít phổ biến hơn — đặc biệt cần thiết ở D3 vì `Severity_1` (~0.97% giao dịch) và `Severity_4` (~1.29% giao dịch) là các mức hiếm, chỉ xuất hiện trong frequent itemsets khi hạ ngưỡng support xuống đủ thấp.


In [8]:
from mlxtend.frequent_patterns import apriori, fpgrowth
import pandas as pd
import time

print("✓ Đã import Apriori và FP-Growth")

SUPPORT_HIGH = 0.01
SUPPORT_LOW = 0.005

MAX_LEN = 3

print("=" * 70)
print("CẤU HÌNH KHAI THÁC FREQUENT ITEMSETS")
print("=" * 70)

print(f"Min support cao : {SUPPORT_HIGH}")
print(f"Min support thấp: {SUPPORT_LOW}")
print(f"Max itemset size: {MAX_LEN}")


✓ Đã import Apriori và FP-Growth
CẤU HÌNH KHAI THÁC FREQUENT ITEMSETS
Min support cao : 0.01
Min support thấp: 0.005
Max itemset size: 3


> **MAX_LEN = 3 được sử dụng để giới hạn kích thước itemset, nhằm kiểm soát không gian tìm kiếm và tránh số lượng tổ hợp tăng quá lớn. Do đó, các luật được phân tích trong bài chỉ dựa trên các frequent itemsets có tối đa 3 item. Với D3 chỉ có 30 item (so với 140 ở D2), không gian tìm kiếm nhỏ hơn nhiều nên cả hai thuật toán chạy rất nhanh ngay cả ở min_support = 0.005.**


In [9]:
# CHẠY THUẬT TOÁN APRIORI

print("=" * 70)
print("CHẠY THUẬT TOÁN APRIORI")
print("=" * 70)

apriori_results = {}

for support in [SUPPORT_HIGH, SUPPORT_LOW]:

    print(f"\n--- Apriori | min_support = {support} ---")

    start_time = time.perf_counter()

    frequent = apriori(
        basket_df,
        min_support=support,
        use_colnames=True,
        max_len=MAX_LEN,
        low_memory=True
    )

    elapsed = time.perf_counter() - start_time

    apriori_results[support] = {
        "frequent_itemsets": frequent,
        "time": elapsed
    }

    print(f"Số frequent itemsets: {len(frequent):,}")
    print(f"Thời gian chạy       : {elapsed:.4f} giây\n")

# CHẠY THUẬT TOÁN FP-GROWTH

print("=" * 70)
print("CHẠY THUẬT TOÁN FP-GROWTH")
print("=" * 70)

fpgrowth_results = {}

for support in [SUPPORT_HIGH, SUPPORT_LOW]:

    print(f"\n--- FP-Growth | min_support = {support} ---")

    start_time = time.perf_counter()

    frequent = fpgrowth(
        basket_df,
        min_support=support,
        use_colnames=True,
        max_len=MAX_LEN
    )

    elapsed = time.perf_counter() - start_time

    fpgrowth_results[support] = {
        "frequent_itemsets": frequent,
        "time": elapsed
    }

    print(f"Số frequent itemsets: {len(frequent):,}")
    print(f"Thời gian chạy       : {elapsed:.4f} giây")


CHẠY THUẬT TOÁN APRIORI

--- Apriori | min_support = 0.01 ---
Số frequent itemsets: 490
Thời gian chạy       : 0.2714 giây


--- Apriori | min_support = 0.005 ---
Số frequent itemsets: 726
Thời gian chạy       : 0.3025 giây

CHẠY THUẬT TOÁN FP-GROWTH

--- FP-Growth | min_support = 0.01 ---
Số frequent itemsets: 490
Thời gian chạy       : 0.9735 giây

--- FP-Growth | min_support = 0.005 ---
Số frequent itemsets: 726
Thời gian chạy       : 0.9331 giây


In [10]:
print("=" * 70)
print("VÍ DỤ FREQUENT ITEMSETS")
print("=" * 70)

frequent_apriori = apriori_results[SUPPORT_HIGH]["frequent_itemsets"]

display(
    frequent_apriori
    .sort_values("support", ascending=False)
    .head(20)
)


VÍ DỤ FREQUENT ITEMSETS


,support,itemsets
25,0.889356,frozenset({Weekday_Accident})
2,0.735607,frozenset({Day})
5,0.684138,frozenset({Severity_2})
61,0.658025,"frozenset({Weekday_Accident, Day})"
104,0.625970,"frozenset({Weekday_Accident, Severity_2})"
45,0.508811,"frozenset({Day, Severity_2})"
248,0.468467,"frozenset({Weekday_Accident, Day, Severity_2})"
11,0.424223,frozenset({Time_Morning})
156,0.391253,"frozenset({Time_Morning, Weekday_Accident})"
50,0.365978,"frozenset({Time_Morning, Day})"


### So sánh Apriori và FP-Growth

Hai thuật toán được chạy trên cùng một ma trận basket và cùng các mức `min_support`, do đó có thể so sánh dựa trên:

- Số lượng frequent itemsets tìm được.
- Thời gian thực thi.
- Ảnh hưởng của việc giảm `min_support`.

Về nguyên tắc, Apriori và FP-Growth phải cho cùng tập frequent itemsets khi sử dụng cùng dữ liệu và cùng ngưỡng `min_support`. Sự khác biệt chủ yếu nằm ở phương pháp tìm kiếm và thời gian thực thi.


In [11]:
comparison_rows = []

for support in [SUPPORT_HIGH, SUPPORT_LOW]:

    apriori_data = apriori_results[support]
    fpgrowth_data = fpgrowth_results[support]

    comparison_rows.append({
        "Algorithm": "Apriori",
        "min_support": support,
        "Frequent itemsets": len(apriori_data["frequent_itemsets"]),
        "Runtime (s)": round(apriori_data["time"], 4)
    })

    comparison_rows.append({
        "Algorithm": "FP-Growth",
        "min_support": support,
        "Frequent itemsets": len(fpgrowth_data["frequent_itemsets"]),
        "Runtime (s)": round(fpgrowth_data["time"], 4)
    })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

print("=" * 70)
print("SO SÁNH ẢNH HƯỞNG CỦA MIN_SUPPORT")
print("=" * 70)

for algorithm, results in [
    ("Apriori", apriori_results),
    ("FP-Growth", fpgrowth_results)
]:

    n_high = len(results[SUPPORT_HIGH]["frequent_itemsets"])
    n_low = len(results[SUPPORT_LOW]["frequent_itemsets"])

    increase = n_low - n_high
    increase_pct = increase / n_high * 100

    print(f"\n{algorithm}")
    print(f"  min_support = {SUPPORT_HIGH}: {n_high:,} itemsets")
    print(f"  min_support = {SUPPORT_LOW}: {n_low:,} itemsets")
    print(f"  Tăng thêm             : {increase:,} itemsets")
    print(f"  Tỷ lệ tăng            : {increase_pct:.2f}%")


,Algorithm,min_support,Frequent itemsets,Runtime (s)
0,Apriori,0.010,490,0.2714
1,FP-Growth,0.010,490,0.9735
2,Apriori,0.005,726,0.3025
3,FP-Growth,0.005,726,0.9331


SO SÁNH ẢNH HƯỞNG CỦA MIN_SUPPORT

Apriori
  min_support = 0.01: 490 itemsets
  min_support = 0.005: 726 itemsets
  Tăng thêm             : 236 itemsets
  Tỷ lệ tăng            : 48.16%

FP-Growth
  min_support = 0.01: 490 itemsets
  min_support = 0.005: 726 itemsets
  Tăng thêm             : 236 itemsets
  Tỷ lệ tăng            : 48.16%


### Nhận xét

Ở `min_support = 0.01`, cả Apriori và FP-Growth đều tìm được **490 frequent itemsets**. Khi giảm ngưỡng xuống `0.005`, số lượng tăng lên **726 frequent itemsets** (tăng khoảng 48%) — hai thuật toán luôn cho cùng số lượng itemset ở cùng một `min_support`, đúng như output ở cell trên.

So với D2 (3,867 → 9,535 itemsets trên 140 item), D3 chỉ có 30 item nên số lượng frequent itemsets nhỏ hơn nhiều — không gian tổ hợp tỷ lệ với số item, không phải số transaction.

Về thời gian thực thi: quan sát ở output cell trên cho thấy **Apriori nhanh hơn FP-Growth khá rõ** trên tập dữ liệu này (khoảng 3–4 lần) — với không gian chỉ 30 item, phần overhead xây dựng cây FP-tree của FP-Growth không mang lại lợi thế so với việc duyệt trực tiếp của Apriori. Runtime tuyệt đối (giây) dao động nhẹ giữa các lần chạy tùy tải hệ thống, nên số liệu chính xác cần xem trực tiếp ở output cell phía trên thay vì số cố định ghi ở đây; xu hướng tương đối (Apriori nhanh hơn FP-Growth) là ổn định qua các lần chạy thử.

Đây là kết quả thực nghiệm trên dữ liệu và cấu hình hiện tại, không nên suy rộng thành kết luận rằng Apriori luôn nhanh hơn FP-Growth trong mọi trường hợp.

Điểm quan trọng là cả hai thuật toán cho cùng số lượng frequent itemsets ở cùng một `min_support`, cho thấy kết quả khai thác là nhất quán.


# 3. Độ đo và lựa chọn ngưỡng trong luật kết hợp

Sau khi tìm được các **frequent itemsets**, bước tiếp theo là xây dựng và đánh giá các **association rules (luật kết hợp)**.

Một luật kết hợp có dạng:

$$
A \rightarrow B
$$

Trong đó:

- $A$: tập điều kiện ở vế trái, gọi là **antecedent**.
- $B$: tập điều kiện ở vế phải, gọi là **consequent**.
- $A \cap B = \emptyset$.

Ví dụ:

$$
\{Railway\} \rightarrow \{Crossing\}
$$

có thể hiểu là:

> Khi một vụ tai nạn xảy ra gần đường sắt, vụ tai nạn đó có xu hướng đồng thời xảy ra gần một giao lộ (crossing).

Để đánh giá một luật, ba độ đo quan trọng được sử dụng là:

1. **Support**
2. **Confidence**
3. **Lift**

Ngoài ra, cần lựa chọn các ngưỡng phù hợp để loại bỏ những luật quá yếu hoặc không có ý nghĩa thực tế.


## 3.1. Support
### **Định nghĩa**

**Support** đo mức độ phổ biến của một tập điều kiện trong toàn bộ tập dữ liệu giao dịch.

Đối với luật:

$$
A \rightarrow B
$$

support được tính dựa trên việc **A và B cùng xuất hiện trong một giao dịch**.

Support càng cao nghĩa là sự kết hợp giữa A và B xuất hiện càng thường xuyên trong dữ liệu.

Ví dụ:

Nếu có 100.000 vụ tai nạn và 2.000 vụ chứa đồng thời A và B thì:

$$
Support(A \rightarrow B)
=
\frac{2000}{100000}
=
0.02
$$

hay **2%**.

Điều này có nghĩa là 2% tổng số vụ tai nạn chứa đồng thời A và B.

### **Công thức**

Gọi:

- $N$: tổng số giao dịch.
- $count(A \cup B)$: số giao dịch chứa đồng thời A và B.

Khi đó:

$$
Support(A \rightarrow B)
=
\frac{count(A \cup B)}{N}
$$

Support không phụ thuộc vào hướng của luật.

Do đó:

$$
Support(A \rightarrow B)
=
Support(B \rightarrow A)
$$


## 3.2. Confidence
### **Định nghĩa**

**Confidence** đo xác suất xuất hiện của \(B\) khi \(A\) đã xuất hiện.

Nói đơn giản:

> Trong những vụ tai nạn đã có điều kiện A, có bao nhiêu phần trăm cũng có điều kiện B?

Confidence có tính đến **hướng của luật**.

Do đó:

$$
Confidence(A \rightarrow B)
$$

có thể khác:

$$
Confidence(B \rightarrow A)
$$

Confidence càng cao thì khả năng B xuất hiện khi A xuất hiện càng lớn.

### **Công thức**

Confidence được tính bằng support của A và B chia cho support của A:

$$
Confidence(A \rightarrow B)
=
\frac{Support(A \cup B)}
{Support(A)}
$$

Tương đương:

$$
Confidence(A \rightarrow B)
=
P(B|A)
$$


## 3.3. Lift
### **Định nghĩa**

**Lift** đo mức độ liên kết giữa A và B bằng cách so sánh xác suất xuất hiện B khi A xuất hiện với xác suất B xuất hiện một cách độc lập.

Lift giúp khắc phục một hạn chế của confidence.

Ví dụ, `Severity_2` chiếm khoảng 68% tổng số vụ tai nạn trong dữ liệu — vì vốn đã rất phổ biến, nhiều luật:

$$
A \rightarrow Severity\_2
$$

có thể có confidence cao dù A không thực sự liên quan đến mức độ nghiêm trọng 2. Vì vậy cần sử dụng Lift để kiểm tra xem A và B có thực sự xuất hiện cùng nhau nhiều hơn mức kỳ vọng hay không.

### **Công thức**

Lift được tính:

$$
Lift(A \rightarrow B)
=
\frac{Confidence(A \rightarrow B)}
{Support(B)}
$$

Hoặc:

$$
Lift(A \rightarrow B)
=
\frac{Support(A \cup B)}
{Support(A)\times Support(B)}
$$

### **Ý nghĩa**

**Lift > 1**

A và B có xu hướng xuất hiện cùng nhau nhiều hơn mức kỳ vọng nếu chúng độc lập.

**Lift = 1**

A và B gần như độc lập.

**Lift < 1**

A và B có xu hướng xuất hiện cùng nhau ít hơn mức kỳ vọng.

Do đó, trong khai phá luật kết hợp, các luật có **Lift > 1** thường được quan tâm hơn.


## 3.4. Ví dụ minh họa

Giả sử có 10 giao dịch (vụ tai nạn):

| Transaction | Điều kiện |
|---|---|
| T1 | A, B |
| T2 | A, B |
| T3 | A, B |
| T4 | A |
| T5 | A |
| T6 | B |
| T7 | B |
| T8 | C |
| T9 | C |
| T10 | A, C |

Xét luật:

$$
A \rightarrow B
$$

Ta có:

- A xuất hiện trong 6 giao dịch: T1, T2, T3, T4, T5, T10.
- B xuất hiện trong 5 giao dịch: T1, T2, T3, T6, T7.
- A và B cùng xuất hiện trong 3 giao dịch: T1, T2, T3.

### **Support**

$$
Support(A \rightarrow B)
=
\frac{3}{10}
=
0.3
$$

### **Confidence**

$$
Confidence(A \rightarrow B)
=
\frac{Support(A\cup B)}
{Support(A)}
=
\frac{0.30}{0.60}
=
0.5
$$

hay **50%**.

### **Lift**

$$
Lift(A \rightarrow B)
=
\frac{0.50}{0.50}
=
1
$$

Như vậy, mặc dù confidence đạt 50%, Lift = 1 cho thấy A và B không thể hiện mối liên hệ mạnh hơn mức kỳ vọng nếu chúng độc lập.


In [12]:
# Ví dụ minh họa cách tính Support, Confidence và Lift

total_transactions = 10

count_A = 6
count_B = 5
count_AB = 3

support_AB = count_AB / total_transactions
support_A = count_A / total_transactions
support_B = count_B / total_transactions

confidence_A_to_B = support_AB / support_A
lift_A_to_B = confidence_A_to_B / support_B

print(f"Support(A → B)    = {support_AB:.2f}")
print(f"Confidence(A → B) = {confidence_A_to_B:.2f}")
print(f"Lift(A → B)       = {lift_A_to_B:.2f}")


Support(A → B)    = 0.30
Confidence(A → B) = 0.50
Lift(A → B)       = 1.00


## 3.5. Lựa chọn ngưỡng

Việc lựa chọn ngưỡng cho association rules cần cân bằng giữa:

- **Không bỏ sót những mối quan hệ hữu ích**.
- **Không tạo ra quá nhiều luật yếu hoặc không có ý nghĩa**.

Nếu chọn ngưỡng quá thấp, số lượng luật có thể tăng lên rất lớn.

Ví dụ:

- Support quá thấp → giữ lại nhiều itemset hiếm.
- Confidence quá thấp → giữ lại các luật có khả năng dự đoán yếu.
- Không kiểm soát Lift → có thể giữ lại những luật có confidence cao chỉ vì consequent vốn đã phổ biến (ví dụ `Severity_2`, chiếm ~68% giao dịch).

Khi đó notebook có thể sinh ra **hàng nghìn luật**, nhưng phần lớn không mang lại thông tin hữu ích.

Do đó, trong bài này sử dụng quy trình:

$$
\text{Frequent Itemsets}
\rightarrow
\text{Association Rules}
\rightarrow
\text{Confidence Filter}
\rightarrow
\text{Lift Filter}
$$

### Ngưỡng được lựa chọn

Ta sử dụng cùng ngưỡng với bộ D2 để tiện so sánh:

$$
Confidence \geq 0.5
$$

và:

$$
Lift \geq 1.2
$$

Tức là chỉ giữ các luật thỏa mãn đồng thời:

- Confidence ít nhất **50%**.
- Lift lớn hơn **1.2**.

### Vì sao chọn Confidence = 0.5?

Confidence 50% nghĩa là:

> Khi A xuất hiện, B cũng xuất hiện trong ít nhất một nửa các trường hợp.

Đây là một mức tương đối nghiêm ngặt để loại bỏ các luật có khả năng dự đoán quá thấp.

Tuy nhiên, confidence một mình chưa đủ vì nó có thể cao do B vốn phổ biến (ví dụ B = `Severity_2` hoặc B = `Weekday_Accident`, chiếm phần lớn dữ liệu).

### Vì sao chọn Lift $\geq$ 1.2?

Lift $\geq$ 1.2 yêu cầu A và B có xu hướng xuất hiện cùng nhau nhiều hơn mức kỳ vọng khi chúng độc lập.

Vì vậy kết hợp Confidence $\geq 0.5$ và Lift $\geq 1.2$ giúp tập trung vào các luật vừa có khả năng xuất hiện B khi có A, vừa thể hiện mối liên hệ dương giữa A và B — thay vì các luật có confidence cao chỉ vì B là điều kiện phổ biến (`Severity_2`, `Weekday_Accident`, `Day`).

### Lưu ý về Support

Hai ngưỡng support đã được sử dụng ở phần trước:

- `0.01` — 1%.
- `0.005` — 0.5%.

Ta giữ cả hai mức để đánh giá ảnh hưởng của support đến số lượng luật. Ngưỡng `0.005` thấp hơn đặc biệt quan trọng ở D3 vì đây là ngưỡng cần thiết để các luật liên quan đến `Severity_1`/`Severity_4` (mức hiếm) có cơ hội xuất hiện.


In [13]:
from mlxtend.frequent_patterns import association_rules

# Sinh toàn bộ association rules từ kết quả Apriori
apriori_rules = {}

for support in [SUPPORT_HIGH, SUPPORT_LOW]:
    frequent_itemsets = apriori_results[support]["frequent_itemsets"]

    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=0
    )

    apriori_rules[support] = rules

CONFIDENCE_THRESHOLD = 0.5
LIFT_THRESHOLD = 1.2

filtered_rules = {}

for support in [SUPPORT_HIGH, SUPPORT_LOW]:
    rules = apriori_rules[support]

    filtered = rules[
        (rules["confidence"] >= CONFIDENCE_THRESHOLD) &
        (rules["lift"] >= LIFT_THRESHOLD)
    ].copy()

    filtered_rules[support] = filtered

    print("=" * 70)
    print(f"min_support = {support}")
    print(f"Tổng association rules : {len(rules):,}")
    print(f"Sau khi lọc            : {len(filtered):,}")


min_support = 0.01
Tổng association rules : 2,142
Sau khi lọc            : 114
min_support = 0.005
Tổng association rules : 3,350
Sau khi lọc            : 186


## 3.6. So sánh ảnh hưởng của ngưỡng

Ta so sánh số lượng luật trước và sau khi áp dụng các ngưỡng:

$$
Confidence \geq 0.5
$$

và:

$$
Lift \geq 1.2
$$

Mục đích là kiểm tra xem việc lọc có giúp giảm đáng kể số lượng luật hay không.

Nếu số luật giảm mạnh sau khi lọc, điều đó cho thấy việc sinh tất cả các luật mà không đặt ngưỡng sẽ tạo ra nhiều luật yếu và khó sử dụng trong thực tế.


In [14]:
comparison_rows = []

for support in [SUPPORT_HIGH, SUPPORT_LOW]:

    total_rules = len(apriori_rules[support])
    filtered_count = len(filtered_rules[support])

    removed = total_rules - filtered_count

    removed_pct = (
        removed / total_rules * 100
        if total_rules > 0 else 0
    )

    comparison_rows.append({
        "min_support": support,
        "Association rules": total_rules,
        "Rules after filtering": filtered_count,
        "Rules removed": removed,
        "Removed (%)": round(removed_pct, 2)
    })

rules_comparison_df = pd.DataFrame(comparison_rows)

display(rules_comparison_df)


,min_support,Association rules,Rules after filtering,Rules removed,Removed (%)
0,0.010,2142,114,2028,94.68
1,0.005,3350,186,3164,94.45


## 3.7. So sánh hai mức Support

Hai mức support được sử dụng:

- `min_support = 0.01`
- `min_support = 0.005`

Ở phần trước, kết quả cho thấy:

| min_support | Frequent itemsets |
|---:|---:|
| 0.01 | 490 |
| 0.005 | 726 |

Khi giảm support từ 0.01 xuống 0.005:

$$
726 - 490 = 236
$$

frequent itemsets được phát hiện thêm.

Tỷ lệ tăng:

$$
\frac{236}{490}\times100
\approx48.16\%
$$

Do giảm min_support làm tăng số frequent itemsets, không gian ứng viên để sinh association rules cũng mở rộng, từ đó thường tạo ra nhiều luật hơn — tương tự xu hướng đã quan sát ở D2, dù mức tăng tương đối nhỏ hơn (48% so với 147%) vì không gian item của D3 nhỏ hơn nhiều.

Tuy nhiên, nhiều luật bổ sung có thể là các mối quan hệ hiếm (ví dụ liên quan đến `Severity_1`/`Severity_4`). Vì vậy cần sử dụng confidence và lift để tiếp tục lọc.


In [15]:
support_comparison_rows = []

for support in [SUPPORT_HIGH, SUPPORT_LOW]:

    support_comparison_rows.append({
        "min_support": support,
        "Frequent itemsets": len(
            apriori_results[support]["frequent_itemsets"]
        ),
        "Association rules": len(
            apriori_rules[support]
        ),
        "Rules after filtering": len(
            filtered_rules[support]
        )
    })

support_comparison_df = pd.DataFrame(
    support_comparison_rows
)

display(support_comparison_df)


,min_support,Frequent itemsets,Association rules,Rules after filtering
0,0.010,490,2142,114
1,0.005,726,3350,186


# 4. Lọc luật

Sau khi sinh association rules, số lượng luật có thể rất lớn và không phải luật nào cũng có giá trị phân tích. Một số luật có thể yếu, tầm thường, hiển nhiên hoặc dư thừa.

Do đó, cần áp dụng các tiêu chí lọc dựa trên các độ đo đã trình bày ở phần trước:

- **Support**: loại các luật xuất hiện quá hiếm.
- **Confidence**: loại các luật có khả năng suy diễn thấp.
- **Lift**: loại các luật có mức liên kết gần với độc lập.
- **Tính hiển nhiên**: loại các luật mà consequent gần như được suy ra trực tiếp từ cách xây dựng item ở antecedent (xem mục 4.2).

Mục tiêu của quá trình lọc là giảm số lượng luật nhưng vẫn giữ lại các luật có mức độ phổ biến, liên kết đủ mạnh và **mang thông tin thực sự mới** để phân tích.


## 4.1. Tiêu chí lọc theo ngưỡng định lượng

Trong bài toán này, các luật được lọc theo hai điều kiện chính:

$$
Confidence(A \rightarrow B) \geq 0.5
$$

và:

$$
Lift(A \rightarrow B) \geq 1.2
$$

### Confidence ≥ 0.5

Điều kiện này yêu cầu ít nhất 50% các giao dịch chứa antecedent cũng chứa consequent.

Confidence thấp thường biểu thị khả năng suy diễn yếu và khó sử dụng cho các quyết định dựa trên luật kết hợp.

### Lift ≥ 1.2

Lift lớn hơn 1 cho thấy A và B có xu hướng xuất hiện cùng nhau nhiều hơn trường hợp độc lập.

Sử dụng ngưỡng 1.2 thay vì chỉ `Lift > 1` giúp loại bỏ các luật có mức liên kết chỉ cao hơn độc lập một cách không đáng kể.

Các ngưỡng trên được xem là **ngưỡng thực nghiệm**, được lựa chọn nhằm cân bằng giữa số lượng luật và mức độ hữu ích của luật. Nếu đặt ngưỡng quá thấp, hệ thống có thể sinh ra hàng nghìn luật, trong đó nhiều luật yếu hoặc tầm thường, gây khó khăn cho việc phân tích.


In [16]:
CONFIDENCE_THRESHOLD = 0.5
LIFT_THRESHOLD = 1.2

filtered_rules = {}

for support in [SUPPORT_HIGH, SUPPORT_LOW]:

    rules = apriori_rules[support]

    filtered = rules[
        (rules["confidence"] >= CONFIDENCE_THRESHOLD) &
        (rules["lift"] >= LIFT_THRESHOLD)
    ].copy()

    filtered_rules[support] = filtered

    print(f"min_support = {support}")
    print(f"Số luật sau lọc: {len(filtered):,}")


min_support = 0.01
Số luật sau lọc: 114
min_support = 0.005
Số luật sau lọc: 186


## 4.2. Loại luật hiển nhiên và dư thừa

Ngoài các tiêu chí định lượng, cần hạn chế các luật **hiển nhiên** — tức những luật mà mối liên hệ giữa A và B gần như đã được cài sẵn ngay trong cách xây dựng item, chứ không phải một phát hiện mới từ dữ liệu.

Ở D3, cặp item `time_period` (rời rạc hóa từ `start_hour`) và `Sunrise_Sunset` (`Day`/`Night`) cùng mã hóa một khái niệm gần giống nhau: thời điểm sáng hay tối. Vì vậy, các luật dạng:

$$
\{Time\_Night, ...\} \rightarrow \{Night\}
$$

có Confidence rất cao (~97–99%) gần như theo định nghĩa (khoảng giờ 00–05 gần như luôn được `Sunrise_Sunset` ghi nhận là `Night`), chứ không phản ánh một tri thức mới về tai nạn giao thông. Đây là một luật **hiển nhiên**, cần loại khỏi danh sách luật đáng chú ý dù nó thỏa mãn tiêu chí `Confidence ≥ 0.5` và `Lift ≥ 1.2`.

Quy tắc lọc áp dụng: loại các luật có **consequent là `{Day}` hoặc `{Night}`** trong khi **antecedent chứa một item bắt đầu bằng `Time_`** (tức chứa `time_period`) — vì hai item này đo cùng một hiện tượng sáng/tối theo hai cách rời rạc hóa khác nhau.

Sau khi loại các luật hiển nhiên, các luật còn lại được xếp hạng theo:

1. **Lift giảm dần** – ưu tiên mức độ liên kết mạnh.
2. **Confidence giảm dần** – ưu tiên khả năng suy diễn cao.
3. **Support giảm dần** – ưu tiên luật xuất hiện phổ biến hơn.

Đây là bước xếp hạng và loại bỏ có tiêu chí rõ ràng, không phải loại bỏ dư thừa hoàn toàn cho mọi cặp item có thể tương quan — nhưng đủ để loại các luật gần như tautological trước khi chọn ra các luật nổi bật ở mục 5.


In [17]:
def format_itemset(itemset):
    return ", ".join(sorted(itemset))


def is_obvious_rule(row):
    """
    Luật bị coi là hiển nhiên nếu consequent chỉ là {Day} hoặc {Night}
    trong khi antecedent đã chứa một item time_period (Time_*) - hai item
    này cùng mã hóa khái niệm sáng/tối theo hai cách rời rạc hóa khác nhau.
    """
    consequent = set(row["consequents"])
    antecedent = set(row["antecedents"])

    if consequent in ({"Day"}, {"Night"}):
        if any(item.startswith("Time_") for item in antecedent):
            return True

    return False


ranked_rules = {}
obvious_counts = {}

for support in [SUPPORT_HIGH, SUPPORT_LOW]:

    rules = filtered_rules[support].copy()

    rules["is_obvious"] = rules.apply(is_obvious_rule, axis=1)
    obvious_counts[support] = int(rules["is_obvious"].sum())

    rules = rules[~rules["is_obvious"]].drop(columns="is_obvious")

    rules = rules.sort_values(
        by=["lift", "confidence", "support"],
        ascending=[False, False, False]
    )

    ranked_rules[support] = rules

    print(f"min_support = {support}")
    print(f"  Luật sau lọc ngưỡng        : {len(filtered_rules[support]):,}")
    print(f"  Trong đó bị coi là hiển nhiên: {obvious_counts[support]:,}")
    print(f"  Còn lại sau khi loại hiển nhiên: {len(rules):,}")


for support in [SUPPORT_HIGH, SUPPORT_LOW]:

    print(f"\n{'=' * 70}")
    print(f"TOP LUẬT (đã loại hiển nhiên) | min_support = {support}")
    print("=" * 70)

    display_rules = ranked_rules[support].head(20).copy()

    display_rules["antecedents"] = (
        display_rules["antecedents"].apply(format_itemset)
    )

    display_rules["consequents"] = (
        display_rules["consequents"].apply(format_itemset)
    )

    display(
        display_rules[
            [
                "antecedents",
                "consequents",
                "support",
                "confidence",
                "lift"
            ]
        ]
    )


min_support = 0.01
  Luật sau lọc ngưỡng        : 114
  Trong đó bị coi là hiển nhiên: 39
  Còn lại sau khi loại hiển nhiên: 75
min_support = 0.005
  Luật sau lọc ngưỡng        : 186
  Trong đó bị coi là hiển nhiên: 52
  Còn lại sau khi loại hiển nhiên: 134

TOP LUẬT (đã loại hiển nhiên) | min_support = 0.01


,antecedents,consequents,support,confidence,lift
530,"Traffic_Signal, Weather_Mostly Cloudy",Crossing,0.015650,0.511772,3.829585
1161,Time_Night,"Night, Severity_2",0.051801,0.645479,3.681566
429,Crossing,"Severity_2, Traffic_Signal",0.086145,0.644620,3.608359
1347,Time_Night,"Night, Weekday_Accident",0.066849,0.832983,3.600831
493,"Crossing, Time_Morning",Traffic_Signal,0.044596,0.717136,3.565834
397,"Crossing, Night",Traffic_Signal,0.021389,0.713750,3.548999
535,"Crossing, Weather_Partly Cloudy",Traffic_Signal,0.010718,0.713460,3.547557
543,Crossing,"Traffic_Signal, Weekday_Accident",0.089704,0.671252,3.546873
541,"Crossing, Weekday_Accident",Traffic_Signal,0.089704,0.710996,3.535304
517,"Crossing, Weather_Clear",Traffic_Signal,0.016870,0.710843,3.534545



TOP LUẬT (đã loại hiển nhiên) | min_support = 0.005


,antecedents,consequents,support,confidence,lift
652,"Railway, Severity_2",Crossing,0.006383,0.874738,6.545659
658,"Railway, Weekday_Accident",Crossing,0.006623,0.768365,5.749673
496,"Day, Railway",Crossing,0.005295,0.767012,5.739547
428,"Amenity, Traffic_Signal",Crossing,0.006071,0.754442,5.645484
17,Railway,Crossing,0.007113,0.752973,5.634495
654,Railway,"Crossing, Severity_2",0.006383,0.675676,5.562325
660,Railway,"Crossing, Weekday_Accident",0.006623,0.701081,5.556810
498,Railway,"Crossing, Day",0.005295,0.560541,5.407037
425,Amenity,"Crossing, Severity_2",0.007965,0.566038,4.659759
423,"Amenity, Severity_2",Crossing,0.007965,0.614415,4.597667


# 5. Nhận xét kỹ thuật 4 luật nổi bật

Sau khi lọc theo `Confidence ≥ 0.5`, `Lift ≥ 1.2` và loại các luật hiển nhiên (mục 4.2), các luật còn lại được xếp hạng theo `Lift`, `Confidence` và `Support`. Bốn luật tiêu biểu dưới đây được lựa chọn để thể hiện các khía cạnh khác nhau: Lift rất cao trên tập hiếm, Support lớn trên tập phổ biến, và mối liên hệ giữa hạ tầng — thời tiết — mức độ nghiêm trọng.


### Luật 1: `Railway, Severity_2 → Crossing`

* **Support = 0.006383**
* **Confidence = 0.874738**
* **Lift = 6.545659**

Đây là luật có **Lift cao nhất** trong toàn bộ kết quả (ở cả hai mức support). Confidence ≈ 87.5% cho biết: trong các vụ tai nạn mức độ nghiêm trọng 2 xảy ra gần đường sắt (`Railway`), khoảng 87.5% cũng xảy ra gần một giao lộ (`Crossing`). Lift ≈ 6.55 cho thấy khả năng đồng thời xuất hiện `Crossing` trong nhóm này cao hơn khoảng 6.5 lần so với mức kỳ vọng nếu hai điều kiện độc lập.

**Ý nghĩa kỹ thuật:** đường sắt và giao lộ đường bộ thường được xây dựng gần nhau về mặt hạ tầng (đường ray cắt ngang đường bộ tạo thành giao lộ), nên luật này phần nào phản ánh đặc điểm quy hoạch hạ tầng giao thông chứ không hẳn là một quan hệ nhân quả về hành vi lái xe. Support thấp (0.64%) vì tổ hợp `Railway` + `Severity_2` bản thân đã hiếm (`Railway` chỉ xuất hiện ở ~0.94% tổng số vụ tai nạn), nhưng Lift rất cao cho thấy đây vẫn là một mối liên kết đáng chú ý, không phải nhiễu ngẫu nhiên.


### Luật 2: `Amenity, Traffic_Signal → Crossing`

* **Support = 0.006071**
* **Confidence = 0.754442**
* **Lift = 5.645484**

Confidence ≈ 75.4% nghĩa là phần lớn các vụ tai nạn xảy ra gần tiện ích công cộng (`Amenity`, ví dụ trạm xăng, nhà hàng) và đèn tín hiệu giao thông cũng xảy ra gần một giao lộ. Lift ≈ 5.65 cho thấy mức liên kết mạnh, cao hơn khoảng 5.6 lần so với kỳ vọng độc lập.

**Ý nghĩa kỹ thuật:** tương tự Luật 1, đây là một mối liên hệ chủ yếu do đặc điểm quy hoạch đô thị — các tiện ích công cộng thường được đặt tại khu vực có mật độ giao thông cao, vốn cũng là nơi có đèn tín hiệu và giao lộ. Luật này minh họa rằng luật kết hợp trên dữ liệu tai nạn có thể phát hiện **quan hệ giữa các đặc điểm hạ tầng xung quanh** chứ không chỉ giữa điều kiện thời tiết/thời gian.


### Luật 3: `Crossing → Traffic_Signal, Weekday_Accident`

* **Support = 0.089704**
* **Confidence = 0.671252**
* **Lift = 3.546873**

Khác với hai luật trên, luật này có **Support lớn hơn nhiều (~9%)** — tức tổ hợp điều kiện này xuất hiện trong gần 1 trên 10 vụ tai nạn của toàn bộ mẫu dữ liệu. Confidence ≈ 67.1% cho biết: trong các vụ tai nạn xảy ra tại giao lộ (`Crossing`), khoảng 67.1% đồng thời có đèn tín hiệu giao thông và xảy ra vào ngày trong tuần. Lift ≈ 3.55 cho thấy mối liên kết dương rõ ràng.

**Ý nghĩa kỹ thuật:** đây là luật có sự cân bằng tốt giữa **độ phổ biến (Support lớn)** và **độ mạnh của quan hệ (Lift > 3)**, phản ánh một mô thức hành vi giao thông hợp lý: tai nạn tại giao lộ có đèn tín hiệu tập trung nhiều vào các ngày trong tuần — thời điểm có mật độ giao thông đi làm/đi học cao hơn cuối tuần. Đây là luật có giá trị thực tiễn cao nhất trong 4 luật, vì Support lớn nghĩa là mô thức này ảnh hưởng đến một tỷ lệ đáng kể vụ tai nạn trong dữ liệu, không chỉ là một trường hợp hiếm gặp.


### Luật 4: `Traffic_Signal, Weather_Mostly Cloudy → Crossing`

* **Support = 0.015650**
* **Confidence = 0.511772**
* **Lift = 3.829585**

Confidence ≈ 51.2% — vừa đạt ngưỡng lọc — cho biết hơn một nửa số vụ tai nạn có đèn tín hiệu giao thông và xảy ra khi trời nhiều mây (`Mostly Cloudy`) cũng xảy ra tại giao lộ. Lift ≈ 3.83 cho thấy mối liên kết khá mạnh dù Confidence không quá cao.

**Ý nghĩa kỹ thuật:** luật này minh họa rằng một luật có thể có **Confidence chỉ vừa đủ ngưỡng nhưng Lift vẫn cao**, vì `Crossing` không phải là điều kiện quá phổ biến trong toàn bộ dữ liệu — nên xác suất B xuất hiện một cách độc lập (`Support(B)`) thấp, khiến Lift tăng lên đáng kể ngay cả khi Confidence ở mức vừa phải. Đây cũng là luật duy nhất trong 4 luật kết hợp cả điều kiện thời tiết lẫn hạ tầng, cho thấy thuật toán có thể phát hiện các tổ hợp đa yếu tố (thời tiết + hạ tầng) cùng liên quan đến vị trí xảy ra tai nạn.


### Tổng kết

| Luật | Support | Confidence | Lift | Đặc điểm nổi bật |
|---|---:|---:|---:|---|
| Railway, Severity_2 → Crossing | 0.64% | 87.5% | 6.55 | Lift cao nhất, hạ tầng hiếm |
| Amenity, Traffic_Signal → Crossing | 0.61% | 75.4% | 5.65 | Hạ tầng đô thị, Lift cao |
| Crossing → Traffic_Signal, Weekday_Accident | 8.97% | 67.1% | 3.55 | Support lớn nhất, mô thức đi lại |
| Traffic_Signal, Weather_Mostly Cloudy → Crossing | 1.57% | 51.2% | 3.83 | Kết hợp thời tiết + hạ tầng |

Cả 4 luật đều liên quan đến `Crossing` (giao lộ) ở vế phải hoặc vế trái, cho thấy đặc điểm hạ tầng này có mối liên kết mạnh với nhiều điều kiện khác trong dữ liệu tai nạn — một phát hiện hợp lý về mặt an toàn giao thông (giao lộ là điểm giao cắt luồng phương tiện, vốn tiềm ẩn rủi ro va chạm cao hơn đường thẳng). Ngược lại, các luật liên quan trực tiếp đến `Severity_1`/`Severity_4` (mức hiếm) không xuất hiện trong top luật sau khi lọc — cho thấy trong mẫu dữ liệu này, mức độ nghiêm trọng không có mối liên kết đủ mạnh (Lift ≥ 1.2) với riêng bất kỳ tổ hợp điều kiện thời tiết/hạ tầng/thời gian nào, mà có thể phụ thuộc vào các yếu tố khác không có trong tập item hiện tại (ví dụ tốc độ, loại đường).
